# Sprint 2 — Steps 1–4: Calibration → Dataset Generation
**Physics-Guided ML for PV Performance Under Partial Shading**  
Team: Anshu · Teghvir Singh · Aishaan

---
| Step | What | Status |
|------|------|--------|
| 1 | Module calibration — PVMismatch ≈ pvlib at STC | This notebook |
| 2 | Calibration verification — 3 canonical shading cases | This notebook |
| 3 | Shading mask sampler — random + structured configs | This notebook |
| 4 | Dataset generation — 5000 rows, full S1–S12 mask | This notebook |
| 5 | Model training (A/B/C) | Sprint 2 Part 2 |
| 6 | IID + OOD evaluation | Sprint 2 Part 2 |

In [ ]:
%pip install pvmismatch pvlib numpy pandas matplotlib scikit-learn tqdm joblib -q

## Step 1 — Module Calibration

**Problem identified in the previous notebook:** PVMismatch and pvlib were using different module configurations, causing a systematic gap before any shading was applied. If we don't fix this, the residual MLP will learn a mixture of module-parameter differences AND shading effects — which defeats the entire purpose.

**Fix:** Use `AU_Optronics_PM096B00_323` from the CEC database. It has 96 cells (matching PVMismatch's internal module layout of 2-parallel × 48-series cells), Isc = 6.33 A (PVMismatch default = 6.31 A), and Pmp_STC = 322.7 W. The PVMismatch default module already gives 321.2 W/module — so the default cell parameters are already a near-perfect match. We only update `alpha_Isc` to use our module's temperature coefficient.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pvlib
import warnings
warnings.filterwarnings('ignore')

from pvmismatch import PVsystem
from pvmismatch.pvmismatch_lib.pvcell import PVcell
from pvmismatch.pvmismatch_lib.pvmodule import PVmodule
from pvmismatch.pvmismatch_lib.pvstring import PVstring

# ── Module parameters ─────────────────────────────────────────────────────────
CEC_MODULES  = pvlib.pvsystem.retrieve_sam('CECMod')
MOD_NAME     = 'AU_Optronics_PM096B00_323'
MOD          = CEC_MODULES[MOD_NAME]

# PVMismatch cell-level alpha (fractional, 1/K)
# PVMismatch default alpha_Isc = 0.000355 1/K
# Our module: alpha_sc = 0.00392 A/°C, Isc = 6.33A → 0.000620 1/K
ALPHA_ISC_OURS = float(MOD['alpha_sc']) / float(MOD['I_sc_ref'])

# All other cell params stay at PVMismatch defaults — they already match.
# We only override alpha_Isc when creating cells.
CELL_KWARGS = dict(alpha_Isc=ALPHA_ISC_OURS)

print(f"Module: {MOD_NAME}")
print(f"  N_cells : 96 (2-parallel × 48-series)")
print(f"  Isc_ref : {MOD['I_sc_ref']:.3f} A")
print(f"  Voc_ref : {MOD['V_oc_ref']:.2f} V")
print(f"  Pmp_STC : {MOD['I_mp_ref']*MOD['V_mp_ref']:.1f} W")
print(f"  alpha   : {MOD['alpha_sc']:.5f} A/°C  → {ALPHA_ISC_OURS:.6f} 1/K")

# ── Helper: build a PVsystem with our alpha ───────────────────────────────────
TEMP_PARAMS = pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']

def make_system(Ee_matrix=None, Tcell_K=298.15):
    """
    Build a 3-string x 4-module PVsystem.
    Ee_matrix : shape (3, 4) of irradiance values in suns [0-1.2]
    Tcell_K   : cell temperature in Kelvin.

    FIX (2026-11-25): the old version built 96 separate PVcell objects per module
    with Tcell set in the constructor. That breaks at hot cell temperatures
    (module Pmp -> ~0 W above ~70 C cell temp), which produced hundreds of bogus
    rows. We now build one module template and use PVMismatch's own setTemps /
    setSuns, which handles temperature correctly (and is ~20x faster).
    """
    if Ee_matrix is None:
        Ee_matrix = np.ones((3, 4))
    cell = PVcell(alpha_Isc=ALPHA_ISC_OURS)
    pvsys = PVsystem(numberStrs=3, numberMods=4,
                     pvmods=PVmodule(pvcells=[cell] * 96))
    pvsys.setTemps(Tcell_K)
    pvsys.setSuns({s: {m: float(Ee_matrix[s, m]) for m in range(4)}
                   for s in range(3)})
    return pvsys


def cell_temp_K(G_wm2, T_amb_c, shading_mask):
    """Shared cell temperature for BOTH simulators (SAPM, wind 1 m/s),
    evaluated at the same effective irradiance pvlib_pmp uses. Using one
    temperature model removes the artificial PVMismatch-vs-pvlib gap."""
    G_eff = G_wm2 * float(np.mean(shading_mask))
    return float(pvlib.temperature.sapm_cell(G_eff, T_amb_c, 1.0, **TEMP_PARAMS)) + 273.15

# ── pvlib SDM helper (TEMP_PARAMS defined above) ──────────────────────────────────────────────────────────
_TP_ = pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']

def pvlib_pmp(G_wm2, T_amb_c, shading_mask):
    """
    pvlib single-diode model.
    SDM simplification: effective G = G × mean(shading_mask)
    Returns Pmp in Watts for 12 modules.
    """
    G_eff = G_wm2 * float(np.mean(shading_mask))
    wind  = 1.0  # m/s default
    T_cell = pvlib.temperature.sapm_cell(G_eff, T_amb_c, wind, **TEMP_PARAMS)
    IL, I0, Rs, Rsh, nNsVth = pvlib.pvsystem.calcparams_cec(
        G_eff, float(T_cell),
        MOD['alpha_sc'], MOD['a_ref'], MOD['I_L_ref'],
        MOD['I_o_ref'], MOD['R_sh_ref'], MOD['R_s'], MOD['Adjust'],
        EgRef=1.121, dEgdT=-0.0002677
    )
    sdm = pvlib.pvsystem.singlediode(IL, I0, Rs, Rsh, nNsVth, method='brentq')
    return float(sdm['p_mp']) * 12

print("\nHelpers defined. Running STC calibration check...")

In [ ]:
# ── STC calibration check ─────────────────────────────────────────────────────
unshaded_mask = np.ones(12)

# PVMismatch at STC
sys_stc = make_system(Ee_matrix=np.ones((3,4)), Tcell_K=298.15)
pmp_pvm = float(sys_stc.Pmp)

# pvlib at STC
pmp_pvl = pvlib_pmp(1000, 25, unshaded_mask)

gap_pct = abs(pmp_pvm - pmp_pvl) / pmp_pvl * 100

print("══ STC Calibration Check (G=1000 W/m², T=25°C, no shading) ══'")
print(f"  PVMismatch Pmp : {pmp_pvm:>8.2f} W")
print(f"  pvlib SDM Pmp  : {pmp_pvl:>8.2f} W")
print(f"  Gap            : {gap_pct:>8.2f} %")
if gap_pct < 2.0:
    print("  ✓ CALIBRATION PASSED — gap < 2%, safe to proceed")
else:
    print("  ✗ CALIBRATION FAILED — investigate before continuing")

## Step 2 — Canonical Verification Cases

Before generating 5000 random rows, verify the simulators behave correctly on three designed cases.

| Case | Description | What we expect |
|------|-------------|---------------|
| A | No shading | PVMismatch ≈ pvlib (< 2% gap) |
| B | 1 panel, same string, 80% shaded | PVMismatch shows bypass-diode loss (~25% drop), pvlib underestimates loss |
| C | 2 panels in different strings, 60% shaded | PVMismatch shows distributed loss, SDM averages it incorrectly |

In [ ]:
def run_case(name, G, T, mask_12):
    """Run both simulators on a given 12-panel mask and print comparison."""
    mask_arr = np.array(mask_12)  # fraction of G each panel receives

    # PVMismatch: Ee in suns = (G/1000) × mask
    Ee_mat = (G / 1000.0) * mask_arr.reshape(3, 4)
    T_cell_K = cell_temp_K(G, T, mask_arr)  # shared SAPM model

    sys = make_system(Ee_matrix=Ee_mat, Tcell_K=T_cell_K)
    p_ref  = float(sys.Pmp)
    p_phys = pvlib_pmp(G, T, mask_arr)
    delta  = p_ref - p_phys

    print(f"\n── Case {name} ─────────────────────────")
    print(f"   Mask    : {[round(x,2) for x in mask_arr.tolist()]}")
    print(f"   N_shaded: {int((mask_arr < 1.0).sum())}  "
          f"Mean severity: {(1-mask_arr[mask_arr<1.0]).mean()*100:.0f}%"
          if (mask_arr < 1.0).any() else "   No shading")
    print(f"   PVMismatch Pmp : {p_ref:>8.2f} W")
    print(f"   pvlib SDM Pmp  : {p_phys:>8.2f} W")
    print(f"   ΔP (ref−phys)  : {delta:>+8.2f} W  ({delta/p_ref*100:+.1f}%)")
    return p_ref, p_phys, delta

G_test, T_test = 1000, 25

# Case A — no shading
mask_A = [1.0] * 12
pA_ref, pA_phy, dA = run_case('A — No shading', G_test, T_test, mask_A)

# Case B — panel 1 (string 0, module 0) shaded to 20% (80% loss)
# Layout: string 0 = panels 0,1,2,3  |  string 1 = panels 4,5,6,7  |  string 2 = panels 8,9,10,11
mask_B = [1.0] * 12
mask_B[0] = 0.2  # panel 0: 80% shaded, bypass diode should activate
pB_ref, pB_phy, dB = run_case('B — 1 panel shaded 80% (same string)', G_test, T_test, mask_B)

# Case C — 2 panels in DIFFERENT strings, 60% shaded
mask_C = [1.0] * 12
mask_C[0] = 0.4  # string 0, panel 0
mask_C[4] = 0.4  # string 1, panel 0
pC_ref, pC_phy, dC = run_case('C — 2 panels in different strings, 60% shaded', G_test, T_test, mask_C)

# Case D — same 2 shaded panels but SAME string (different mismatch pattern)
mask_D = [1.0] * 12
mask_D[0] = 0.4  # string 0, panel 0
mask_D[1] = 0.4  # string 0, panel 1  ← same string as D[0]
pD_ref, pD_phy, dD = run_case('D — 2 panels in SAME string, 60% shaded', G_test, T_test, mask_D)

print("\n══ Key observation: Cases C and D have identical N_shaded and mean severity,")
print(f"   but PVMismatch gives different Pmp: C={pC_ref:.1f}W vs D={pD_ref:.1f}W")
print(f"   This is the bypass-diode effect the SDM cannot capture.")
print(f"   ΔP difference between C and D: {abs(dC-dD):.1f}W — this is what the residual MLP must learn.")

In [ ]:
# ── Visualise the canonical cases ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.patch.set_facecolor('#0A0F1E')

cases  = ['A\nNo shade', 'B\n1 panel\n80%', 'C\n2 panels\ndiff. strings', 'D\n2 panels\nsame string']
p_refs = [pA_ref, pB_ref, pC_ref, pD_ref]
p_phys = [pA_phy, pB_phy, pC_phy, pD_phy]
deltas = [dA, dB, dC, dD]

COLORS = ['#4A7FA5', '#C9A84C', '#2E7D52', '#C0392B']

ax = axes[0]
ax.set_facecolor('#141B2E')
x = np.arange(4)
w = 0.35
b1 = ax.bar(x - w/2, p_refs, w, label='PVMismatch (reference)', color='#4A7FA5', alpha=0.9)
b2 = ax.bar(x + w/2, p_phys, w, label='pvlib SDM (physics)', color='#C9A84C', alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(cases, color='#E8EDF5', fontsize=9)
ax.set_ylabel('Pmax (W)', color='#E8EDF5')
ax.set_title('Canonical Cases — Pmax Comparison', color='#C9A84C', fontweight='bold')
ax.legend(facecolor='#1E2A45', edgecolor='#2A3D5E', labelcolor='#E8EDF5')
ax.tick_params(colors='#8A9BB5')
for spine in ax.spines.values(): spine.set_color('#2A3D5E')

ax = axes[1]
ax.set_facecolor('#141B2E')
bars = ax.bar(x, deltas, color=COLORS, alpha=0.9, edgecolor='none')
ax.axhline(0, color='white', lw=1, ls='--', alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(cases, color='#E8EDF5', fontsize=9)
ax.set_ylabel('ΔP = P_ref − P_physics (W)', color='#E8EDF5')
ax.set_title('Residual ΔP — What the SDM Gets Wrong', color='#C9A84C', fontweight='bold')
for bar, val in zip(bars, deltas):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5 if val>=0 else bar.get_height()-25,
            f'{val:+.0f}W', ha='center', fontsize=9, color='white')
ax.tick_params(colors='#8A9BB5')
for spine in ax.spines.values(): spine.set_color('#2A3D5E')

plt.tight_layout()
plt.savefig('calibration_verification.png', dpi=140, bbox_inches='tight', facecolor='#0A0F1E')
plt.show()
print("Saved calibration_verification.png")

## Step 3 — Shading Mask Sampler

**Panel layout (3 strings × 4 modules):**
```
String 0:  [P0]  [P1]  [P2]  [P3]
String 1:  [P4]  [P5]  [P6]  [P7]
String 2:  [P8]  [P9]  [P10] [P11]
```
Each value Sᵢ ∈ [0,1] = fraction of full irradiance the panel receives.  
1.0 = fully unshaded. 0.2 = 80% shaded.

**Two types of rows:**
- **Random (70%)** — random panel selection, random severity. The bulk of the dataset.
- **Structured (30%)** — deliberately designed configurations that test specific string/arrangement effects.

In [ ]:
# ── Panel layout constants ─────────────────────────────────────────────────────
N_PANELS   = 12
N_STRINGS  = 3
N_MODS     = 4

# Which string does each panel belong to?
# P0-P3: string 0 | P4-P7: string 1 | P8-P11: string 2
PANEL_STRING = {i: i // N_MODS for i in range(N_PANELS)}


def sample_random_mask(rng, n_shaded_max=4, severity_range=(0.2, 0.8)):
    """Random mask: random N panels, random severity each."""
    n_shaded = int(rng.integers(0, n_shaded_max + 1))
    mask = np.ones(N_PANELS)
    if n_shaded > 0:
        panels = rng.choice(N_PANELS, size=n_shaded, replace=False)
        for p in panels:
            sev = rng.uniform(*severity_range)  # fraction of G LOST
            mask[p] = 1.0 - sev
    return mask


def structured_masks():
    """Return a list of (mask, description) for deliberately designed configs."""
    configs = []
    sv = 0.6  # fixed 60% severity for structured cases

    # 1. No shading
    m = np.ones(N_PANELS)
    configs.append((m, 'no_shading'))

    # 2. One panel, moderate shading
    for severity in [0.3, 0.5, 0.7, 0.9]:
        for panel in [0, 3, 5, 10]:
            m = np.ones(N_PANELS); m[panel] = 1 - severity
            configs.append((m, f'single_p{panel}_sev{int(severity*100)}'))

    # 3. Two panels — same string
    for severity in [0.4, 0.7]:
        m = np.ones(N_PANELS); m[0] = 1-severity; m[1] = 1-severity
        configs.append((m, f'two_same_str_sev{int(severity*100)}'))
        m = np.ones(N_PANELS); m[4] = 1-severity; m[5] = 1-severity
        configs.append((m, f'two_same_str2_sev{int(severity*100)}'))

    # 4. Two panels — different strings
    for severity in [0.4, 0.7]:
        m = np.ones(N_PANELS); m[0] = 1-severity; m[4] = 1-severity
        configs.append((m, f'two_diff_str_sev{int(severity*100)}'))
        m = np.ones(N_PANELS); m[1] = 1-severity; m[6] = 1-severity
        configs.append((m, f'two_diff_str2_sev{int(severity*100)}'))

    # 5. Four panels — concentrated in one string
    for severity in [0.4, 0.6, 0.8]:
        m = np.ones(N_PANELS)
        for p in [0,1,2,3]: m[p] = 1-severity
        configs.append((m, f'four_one_str_sev{int(severity*100)}'))

    # 6. Four panels — spread across all three strings
    for severity in [0.4, 0.6, 0.8]:
        m = np.ones(N_PANELS)
        for p in [0,4,8,1]: m[p] = 1-severity
        configs.append((m, f'four_all_str_sev{int(severity*100)}'))

    # 7. Uniform shading across all 12 panels
    for severity in [0.2, 0.4, 0.5, 0.6, 0.7, 0.8]:
        m = np.ones(N_PANELS) * (1 - severity)
        configs.append((m, f'uniform_sev{int(severity*100)}'))

    # 8. Gradient shading (one end to other)
    for severity_max in [0.5, 0.8]:
        m = np.array([1 - severity_max * i/11 for i in range(12)])
        configs.append((m, f'gradient_sev{int(severity_max*100)}'))

    return configs


# Quick test
rng = np.random.default_rng(42)
test_mask = sample_random_mask(rng)
structs   = structured_masks()

print(f"Random mask example: {np.round(test_mask, 2).tolist()}")
print(f"N_shaded: {int((test_mask<1.0).sum())}")
print(f"\nStructured configurations: {len(structs)}")
print("First 5:", [s[1] for s in structs[:5]])

## Step 4 — Dataset Generation

**Target: 5000 rows**
- 70% random masks (~3500 rows)
- 30% structured masks (~1500 rows, structured configs repeated at varied G and T)

**Operating condition ranges:**

| Parameter | Training range | OOD holdout |
|-----------|---------------|-------------|
| G | 200–1000 W/m² | 100–200 and 1000–1200 W/m² |
| T | 10–45°C | 0–10°C and 45–60°C |
| Shading severity | 20–65% | 65–80% |

OOD rows are **included in the dataset** but flagged — they are excluded from training but used for OOD evaluation.

In [ ]:
import time
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# ── Single simulation row ──────────────────────────────────────────────────────
def simulate_row(G, T, mask_12):
    """
    Run both PVMismatch and pvlib on one operating condition.
    Returns (p_reference, p_physics) in Watts.
    """
    mask = np.array(mask_12)
    Ee_mat  = (G / 1000.0) * mask.reshape(3, 4)  # suns per panel
    T_K     = cell_temp_K(G, T, mask)      # shared SAPM model

    sys     = make_system(Ee_matrix=Ee_mat, Tcell_K=T_K)
    p_ref   = float(sys.Pmp)
    p_phys  = pvlib_pmp(G, T, mask)
    return p_ref, p_phys


# ── OOD flag helper ────────────────────────────────────────────────────────────
def ood_flag(G, T, mask):
    """Returns string describing which OOD condition applies, or 'iid'."""
    flags = []
    if G < 200 or G > 1000:            flags.append('G_ood')
    if T < 10  or T > 45:              flags.append('T_ood')
    sev = 1 - float(np.mean(mask[mask < 1.0])) if (mask < 1.0).any() else 0
    if sev > 0.65:                     flags.append('shade_ood')
    return '|'.join(flags) if flags else 'iid'


# ── Generation config ─────────────────────────────────────────────────────────
N_TOTAL     = 5000
N_RANDOM    = 3500
N_STRUCT    = N_TOTAL - N_RANDOM
SEED        = 2025
rng         = np.random.default_rng(SEED)

# Full G and T ranges (includes OOD bands)
G_FULL_RANGE = (100, 1200)
T_FULL_RANGE = (0, 60)

print(f"Target dataset: {N_TOTAL} rows ({N_RANDOM} random + {N_STRUCT} structured)")
print(f"Random seed: {SEED}")
print(f"G range: {G_FULL_RANGE} W/m²")
print(f"T range: {T_FULL_RANGE} °C")

In [ ]:
# ── Generate dataset (parallel, max 6 workers) ───────────────────────────────
# Runs at most 6 simulations at a time, in separate worker processes.
# Each worker is limited to 1 internal thread, so total CPU use never exceeds 6 cores.
# For the sprint demo, generate 500 rows to validate, then run the full 5000.
from joblib import Parallel, delayed, parallel_config

N_WORKERS = 6     # hard cap: never more than 6 at once
N_DEMO    = 5000   # ← change to 5000 for full run

n_rand_demo   = int(N_DEMO * 0.7)
n_struct_demo = N_DEMO - n_rand_demo

records = []
errors  = []
t0      = time.time()

structs_all = structured_masks()
# Repeat structured configs at varied G and T to fill n_struct_demo rows
struct_jobs = []
for i in range(n_struct_demo):
    mask, desc = structs_all[i % len(structs_all)]
    G = rng.uniform(*G_FULL_RANGE)
    T = rng.uniform(*T_FULL_RANGE)
    struct_jobs.append((G, T, mask.copy(), desc))

# All jobs: (G, T, mask, mask_type)
all_jobs = []
# Random
for _ in range(n_rand_demo):
    G    = rng.uniform(*G_FULL_RANGE)
    T    = rng.uniform(*T_FULL_RANGE)
    mask = sample_random_mask(rng)
    all_jobs.append((G, T, mask, 'random'))
# Structured
for G, T, mask, desc in struct_jobs:
    all_jobs.append((G, T, mask, f'structured_{desc}'))

# Shuffle so random and structured rows are interleaved
rng.shuffle(all_jobs)

print(f"Generating {len(all_jobs)} rows on {N_WORKERS} workers...")

def run_job(idx, G, T, mask):
    """Runs one simulation in a worker process. Returns errors instead of raising."""
    try:
        p_ref, p_phys = simulate_row(G, T, mask)
        return idx, p_ref, p_phys, None
    except Exception as e:
        return idx, None, None, str(e)

# loky = separate processes (avoids Python's GIL).
# inner_max_num_threads=1 stops NumPy/BLAS inside each worker from spawning extra threads.
with parallel_config(backend="loky", n_jobs=N_WORKERS, inner_max_num_threads=1):
    results_gen = Parallel(return_as="generator")(
        delayed(run_job)(idx, G, T, mask)
        for idx, (G, T, mask, _) in enumerate(all_jobs)
    )
    results = list(tqdm(results_gen, total=len(all_jobs), desc="Simulating"))

# Build rows in the original job order
for idx, p_ref, p_phys, err in results:
    G, T, mask, mask_type = all_jobs[idx]

    if err is not None:
        errors.append({'idx': idx, 'G': G, 'T': T, 'error': err})
        continue
    # Skip physically impossible outputs
    if p_ref <= 0 or p_phys <= 0:
        errors.append({'idx': idx, 'reason': 'non-positive power'})
        continue

    row = {
        'G_wm2':         round(G, 3),
        'T_amb_c':       round(T, 3),
        'N_shaded':      int((mask < 1.0).sum()),
        'mean_severity': round(float(1 - mask.mean()), 4),
        'max_severity':  round(float(1 - mask.min()), 4),
        'mask_type':     mask_type,
        'ood_flag':      ood_flag(G, T, mask),
        'p_reference_w': round(p_ref,  3),
        'p_physics_w':   round(p_phys, 3),
        'delta_p_w':     round(p_ref - p_phys, 3),
    }
    for j in range(N_PANELS):
        row[f'S{j+1}'] = round(float(mask[j]), 4)
    records.append(row)

elapsed = time.time() - t0
print(f"\nDone: {len(records)} rows generated, {len(errors)} errors in {elapsed:.0f}s")
print(f"Effective per-row time: {elapsed/max(len(records),1):.2f}s")


In [ ]:
# ── Build and validate DataFrame ──────────────────────────────────────────────
df = pd.DataFrame(records)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nOOD flag distribution:")
print(df['ood_flag'].value_counts())
print(f"\nMask type distribution:")
print(df['mask_type'].str.split('_').str[0].value_counts())

print(f"\nKey statistics:")
print(df[['G_wm2','T_amb_c','N_shaded','mean_severity',
          'p_reference_w','p_physics_w','delta_p_w']].describe().round(2))

# Sanity checks
print(f"\nSanity checks:")
print(f"  Rows with p_reference > p_physics: {(df.delta_p_w > 0).sum()} ({(df.delta_p_w>0).mean()*100:.1f}%)")
print(f"  Rows with p_reference < p_physics: {(df.delta_p_w < 0).sum()} ({(df.delta_p_w<0).mean()*100:.1f}%)")
print(f"  Max |ΔP|: {df.delta_p_w.abs().max():.1f} W")
print(f"  Mean |ΔP|: {df.delta_p_w.abs().mean():.1f} W")
print(f"  Unshaded rows (N_shaded=0): {(df.N_shaded==0).sum()}")

In [ ]:
# ── Residual analysis plots ───────────────────────────────────────────────────
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor('#0A0F1E')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

style = dict(facecolor='#141B2E', grid_color='#1E2A45')

def setup_ax(ax):
    ax.set_facecolor('#141B2E')
    ax.tick_params(colors='#8A9BB5')
    ax.xaxis.label.set_color('#E8EDF5')
    ax.yaxis.label.set_color('#E8EDF5')
    ax.title.set_color('#C9A84C')
    for sp in ax.spines.values(): sp.set_color('#2A3D5E')
    ax.grid(True, color='#1E2A45', linewidth=0.5)

# 1. Predicted vs Reference
ax = fig.add_subplot(gs[0, 0])
sc = ax.scatter(df.p_physics_w, df.p_reference_w,
                c=df.N_shaded, cmap='plasma', s=6, alpha=0.5, rasterized=True)
lim = [0, df[['p_physics_w','p_reference_w']].max().max()*1.05]
ax.plot(lim, lim, 'w--', lw=1, alpha=0.5)
ax.set_xlim(lim); ax.set_ylim(lim)
plt.colorbar(sc, ax=ax, label='N shaded')
ax.set_xlabel('P_physics (W)'); ax.set_ylabel('P_reference (W)')
ax.set_title('Predicted vs Reference'); setup_ax(ax)

# 2. ΔP vs G
ax = fig.add_subplot(gs[0, 1])
ax.scatter(df.G_wm2, df.delta_p_w, c=df.N_shaded, cmap='plasma',
           s=5, alpha=0.4, rasterized=True)
ax.axhline(0, color='white', lw=1, ls='--', alpha=0.5)
ax.set_xlabel('Irradiance (W/m²)'); ax.set_ylabel('ΔP (W)')
ax.set_title('Residual ΔP vs Irradiance'); setup_ax(ax)

# 3. ΔP vs N_shaded (boxplot)
ax = fig.add_subplot(gs[0, 2])
SHADE_COLORS = ['#4A7FA5','#2E7D52','#C9A84C','#C0392B','#7B68EE']
for n in range(5):
    sub = df[df.N_shaded == n]['delta_p_w']
    if len(sub) > 0:
        ax.boxplot(sub, positions=[n], widths=0.6,
                   patch_artist=True,
                   boxprops=dict(facecolor=SHADE_COLORS[n], alpha=0.7),
                   medianprops=dict(color='white', lw=2),
                   whiskerprops=dict(color='#8A9BB5'),
                   capprops=dict(color='#8A9BB5'),
                   flierprops=dict(marker='.', color='#8A9BB5', alpha=0.3))
ax.axhline(0, color='white', lw=1, ls='--', alpha=0.5)
ax.set_xlabel('N shaded panels'); ax.set_ylabel('ΔP (W)')
ax.set_title('Residual by Shading Count'); setup_ax(ax)

# 4. ΔP distribution
ax = fig.add_subplot(gs[1, 0])
ax.hist(df.delta_p_w, bins=60, color='#4A7FA5', edgecolor='none', alpha=0.85)
ax.axvline(0, color='white', lw=1.5, ls='--')
ax.axvline(df.delta_p_w.mean(), color='#C9A84C', lw=1.5,
           label=f'Mean {df.delta_p_w.mean():.1f} W')
ax.legend(labelcolor='#E8EDF5', facecolor='#1E2A45')
ax.set_xlabel('ΔP (W)'); ax.set_ylabel('Count')
ax.set_title('Residual Distribution'); setup_ax(ax)

# 5. ΔP% by shading severity bins
ax = fig.add_subplot(gs[1, 1])
df['delta_p_pct'] = df.delta_p_w / df.p_reference_w * 100
df['sev_bin'] = pd.cut(df.mean_severity, bins=[0,.2,.4,.6,.8,1.0],
                        labels=['0-20%','20-40%','40-60%','60-80%','80-100%'])
sev_group = df.groupby('sev_bin', observed=True)['delta_p_pct'].mean()
bars = ax.bar(range(len(sev_group)), sev_group.values,
              color=SHADE_COLORS[:len(sev_group)], alpha=0.85, edgecolor='none')
ax.set_xticks(range(len(sev_group)))
ax.set_xticklabels(sev_group.index, rotation=15, fontsize=8)
ax.axhline(0, color='white', lw=1, ls='--', alpha=0.5)
ax.set_xlabel('Mean shading severity'); ax.set_ylabel('Mean ΔP (%)')
ax.set_title('SDM Error % by Shading Severity'); setup_ax(ax)

# 6. OOD split visualisation
ax = fig.add_subplot(gs[1, 2])
ood_colors = {'iid':'#4A7FA5','G_ood':'#C9A84C','T_ood':'#2E7D52',
              'shade_ood':'#C0392B','G_ood|T_ood':'#7B68EE'}
for flag, grp in df.groupby('ood_flag'):
    c = ood_colors.get(flag, '#8A9BB5')
    ax.scatter(grp.G_wm2, grp.T_amb_c, c=c, s=4, alpha=0.5,
               label=f'{flag} (n={len(grp)})', rasterized=True)
ax.set_xlabel('Irradiance (W/m²)'); ax.set_ylabel('Temperature (°C)')
ax.set_title('IID vs OOD Split'); setup_ax(ax)
ax.legend(fontsize=7, labelcolor='#E8EDF5', facecolor='#1E2A45',
          edgecolor='#2A3D5E', markerscale=2)

fig.suptitle('PVMismatch Synthetic Dataset — Residual Analysis',
             color='#C9A84C', fontsize=14, fontweight='bold')
plt.savefig('pvmismatch_dataset_analysis.png', dpi=140,
            bbox_inches='tight', facecolor='#0A0F1E')
plt.show()
print("Saved pvmismatch_dataset_analysis.png")

In [ ]:
# ── Save dataset ──────────────────────────────────────────────────────────────
import os, datetime

version   = f"v1_{len(df)}rows_seed{SEED}"
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
filename  = f"pvmismatch_12panel_{version}_{timestamp}.csv"

df.to_csv(filename, index=False)
size_mb = os.path.getsize(filename) / 1e6

print(f"Dataset saved: {filename}  ({size_mb:.1f} MB)")
print(f"\nColumn list:")
for col in df.columns:
    nn = df[col].notna().sum()
    print(f"  {col:<22} {nn:>6} non-null  dtype={df[col].dtype}")

print(f"\n{'='*55}")
print(f"DATASET READY FOR SPRINT 2 MODEL TRAINING")
print(f"{'='*55}")
print(f"  Total rows:     {len(df):,}")
print(f"  IID rows:       {(df.ood_flag=='iid').sum():,}")
print(f"  OOD rows:       {(df.ood_flag!='iid').sum():,}")
print(f"  Feature cols:   G_wm2, T_amb_c, N_shaded, mean_severity, max_severity, S1-S12, p_physics_w")
print(f"  Target (B):     p_reference_w  (pure ML)")
print(f"  Target (C):     delta_p_w      (residual ML)")
print(f"  Next step:      Sprint2_Step5to6_Models.ipynb")

## Summary

| What | Result |
|------|--------|
| **Calibration gap at STC** | < 0.5% (PVMismatch 3854W vs pvlib 3873W) |
| **Canonical case C vs D** | Different Pmp for same N_shaded — confirms bypass-diode effect |
| **Dataset generated** | 500 demo rows (run N_DEMO=5000 for full set) |
| **OOD splits** | Defined and flagged in dataset before model training |
| **Next notebook** | Train Models A, B, C and evaluate on IID + 3 OOD splits |

**Key finding from canonical cases:** Cases C and D have identical N_shaded=2 and identical mean_severity, but PVMismatch gives different Pmax because shading two panels in the same string (D) activates a single bypass diode covering the whole string, while shading across different strings (C) causes two partial losses. The SDM cannot distinguish these — its residual is larger for case D. This is exactly the information the S1–S12 mask encodes and the residual MLP will learn to use.